In [2]:
# import necessary libraries
import pandas as pd
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import glob
# set empty values to skyblue
sns.set(rc={'axes.facecolor':'skyblue'})

# set png resolution
plt.rcParams['figure.dpi'] = 300

KeyboardInterrupt: 

In [4]:
import os
os.chdir('C:/Users/edwin/OneDrive/Documents/GitHub/CHC')

In [20]:

def compute_statistics(csv_path, region, model, season):
    df = pd.read_csv(csv_path, index_col=0)
    df = df.dropna(subset=[season])
    # take the spatial means
    df = (df  # Use the ensemble_means DataFrame
                              .groupby(['time', 'lead_time', season])[['predicted_precip', 'precip']]
                              .mean().reset_index())
    df = df[['predicted_precip', 'precip', season]]

    
    corr = df.groupby([season]).corr(method='spearman').drop(['precip'], axis = 1).iloc[::-2].reset_index()
    corr = corr.rename(columns = {'predicted_precip': 'corr'}).drop(columns = ['level_1'])
    stat = df.groupby([season]).agg(['mean', 'std']).reset_index()
    stat.columns = [season, 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

    # Merging stat and spatial_means_corr to get 1 df with all values
    stat_clean = stat.merge(corr, left_on=[season], right_on=[season], how='left').dropna()
    # Calculating metrics
    stat_clean['potential_skill'] = np.square(stat_clean['corr'])
    stat_clean['conditional_bias'] = np.square(stat_clean['corr'] - (stat_clean['pred_std'] / stat_clean['actual_std']))
    stat_clean['unconditional_bias'] = np.square((stat_clean['pred_mean'] - stat_clean['actual_mean']) / stat_clean['actual_std'])
    stat_clean['skill_score'] = stat_clean['potential_skill'] - stat_clean['conditional_bias'] - stat_clean['unconditional_bias']

    stat_clean = stat_clean.drop(['pred_mean','pred_std','actual_mean','actual_std'], axis = 1)
    stat_clean['region'] = region
    stat_clean['model'] = model
    stat_clean[['season', 'lead_category']] = stat_clean[season].str.split('_', expand = True)  # Extract lead category from the season name   
    stat_clean = stat_clean.drop(columns = [season])
    return stat_clean


In [25]:
# compute the multi model ensemble statistics by first merging all the csv files
dfs_dict = {}

list_of_files = glob.glob('data/seasonal/seasonal_average/*.csv')
for f in list_of_files:
    df = pd.read_csv(f, index_col = 0)
    region = f.split('/')[-1].split('_')[0:-5] # Extract region name
    region = '_'.join(region)
    region = region.split("\\",1)[1]
    df = (df
                            .groupby(['year', 'season', 'lead_category', 'model'])[['predicted_precip', 'precip']]
                            .mean().reset_index())
    df['region'] = region
    df = df.reset_index()
    dfs_dict[f] = df

mme = pd.concat(dfs_dict.values(), ignore_index=True)
mme_mean = mme.groupby(['region', 'lead_category', 'season', 'year'])[['predicted_precip', 'precip']].mean().reset_index()
mme_corr = mme_mean.groupby(['region', 'lead_category', 'season'])[['predicted_precip', 'precip']].corr(method='spearman').drop(['precip'], axis = 1).iloc[::-2].reset_index()
mme_corr = mme_corr.drop(columns = ['level_3']).rename(columns = {'predicted_precip': 'corr'})
mme_corr
mme_stat = mme.groupby(['lead_category', 'region', 'season'])[['predicted_precip', 'precip']].agg(['mean', 'std']).reset_index()
mme_stat.columns = ['lead_category', 'region', 'season', 'pred_mean', 'pred_std', 'actual_mean', 'actual_std']

# Merging stat and spatial_means_corr to get 1 df with all values
mme_stat_clean = mme_stat.merge(mme_corr, left_on=['lead_category', 'region', 'season'], right_on=['lead_category', 'region', 'season'], how='left').dropna()
# Calculating metrics
mme_stat_clean['potential_skill'] = np.square(mme_stat_clean['corr'])
mme_stat_clean['conditional_bias'] = np.square(mme_stat_clean['corr'] - (mme_stat_clean['pred_std'] / mme_stat_clean['actual_std']))
mme_stat_clean['unconditional_bias'] = np.square((mme_stat_clean['pred_mean'] - mme_stat_clean['actual_mean']) / mme_stat_clean['actual_std'])
mme_stat_clean['skill_score'] = mme_stat_clean['potential_skill'] - mme_stat_clean['conditional_bias'] - mme_stat_clean['unconditional_bias']

mme_stat_clean = mme_stat_clean.drop(['pred_mean','pred_std','actual_mean','actual_std'], axis = 1)
mme_stat_clean['model'] = 'mme'  # Extract lead category from the season name   

In [26]:
# Set dictionary of regions and their respective seasons of interest
seasons = {
    'eastern_east_africa':['MAM','OND'],
    'lake_victoria_basin':['DJF','MAM', 'SON'],
    'west_africa':['JAS'],
    'south_sudan':['MJJ','JAS','ASO'],
    'eastern_ukraine':['DJF','AMJ','JA'],
    'southern_africa':['DJF','FMA'],
    'sri_lanka':['OND']
}

# Initialize an empty dictionary to store DataFrames
dfs_dict = {}

# initiate file list
# list_of_files = glob.glob('/content/drive/My Drive/capstone_data/netCDF/*')
list_of_files = glob.glob('data/seasonal/*.csv')
# Loop over all files
for f in list_of_files:
    # Generate the DataFrame
    model = f.split('/')[-1].split('_')[-3] # Extract model name
    region = f.split('/')[-1].split('_')[0:-3] # Extract region name
    region = '_'.join(region)
    region = region.split("\\",1)[1]
    
    for season in seasons[region]:
        df = compute_statistics(f, region, model, season)
        
        # Store the DataFrame in the dictionary with the year as key
        dfs_dict[f'{f}_{season}'] = df

# Concatenate all DataFrames in the dictionary into one DataFrame
final_df = pd.concat(dfs_dict.values(), ignore_index=True)
final_df = pd.concat([final_df, mme_stat_clean], ignore_index=True)

C:\Users\edwin\AppData\Local\Temp\ipykernel_11012\1533938320.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, index_col=0)
C:\Users\edwin\AppData\Local\Temp\ipykernel_11012\1533938320.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, index_col=0)
C:\Users\edwin\AppData\Local\Temp\ipykernel_11012\1533938320.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, index_col=0)
C:\Users\edwin\AppData\Local\Temp\ipykernel_11012\1533938320.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path, index_col=0)
C:\Users\edwin\AppData\Local\Temp\ipykernel_11012\1533938320.py:2: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=Fals

In [27]:
def draw_heatmap(*args, **kwargs):
    data = kwargs.pop('data')
    d = data.pivot(index=args[1], columns=args[0], values=args[2])
    sns.heatmap(d, **kwargs, vmin=0, vmax=1,
                cmap=sns.color_palette('Reds', 10),
                annot=True,
                fmt=".2f",
                linewidths=0.1, linecolor='black')
    #plt.xticks(np.arange(0.5, 12.5, 1))  # Set xticks explicitly
    #plt.gca().set_xticklabels(np.arange(0.5, 12.5, 1))  # Set xticklabels
    plt.xticks(fontsize=7)
    plt.yticks(fontsize=7)
    plt.gca().invert_yaxis()
    plt.gca().invert_xaxis()
    

fg = sns.FacetGrid(final_df, col='region', row='model', sharex=False, sharey=False)
fg.map_dataframe(draw_heatmap, 'lead_category', 'season', 'potential_skill', square = True)
fg.set_titles('Potential Skill \n Region={col_name} \n Model={row_name}')
fg.set_ylabels("Season")
fg.set_xlabels("Lead Category")
fg.savefig('figures/seasonal_potential_skill.png')
plt.close()

In [39]:
def draw_barchart(*args, **kwargs):
    data = kwargs.pop('data')
    # Group by lead_category to compute mean metrics
    df_grouped = data.groupby('lead_category')[['potential_skill', 'conditional_bias']].mean().reset_index()
    categories = df_grouped['lead_category']
    x = np.arange(len(categories))
    width = 0.35
    ax = plt.gca()
    # Define colors for consistency across facets
    colors = sns.color_palette("deep", 2)
    # Plot bars: potential skill on left, conditional bias on right
    ax.bar(x - width/2, df_grouped['potential_skill'], width, 
           color=colors[0])
    ax.bar(x + width/2, df_grouped['conditional_bias'], width, 
           color=colors[1])
    # Label the x-axis using the lead_category values
    ax.set_xticks(x)
    ax.set_xticklabels(categories, fontsize=7)
    ax.set_xlabel("Lead Category")
    ax.set_ylabel("Metric Value")
    # Optional: remove invert_xaxis if not desired
    plt.gca().invert_xaxis()

# Create the FacetGrid across 'region' and 'model'
fg = sns.FacetGrid(final_df, col='model', row='region', sharex=True, sharey=True)
fg.map_dataframe(draw_barchart)
fg.set_titles('Region: {row_name} \n Model: {col_name}')
fg.set_axis_labels("Lead Category", "Metric Value")
fg.fig.subplots_adjust(top=0.9)
fg.fig.suptitle('Potential Skill and Conditional Bias by Region, Model, & Lead Category')

# Create a single global legend using the same colors
patch1 = mpatches.Patch(color=sns.color_palette("deep", 2)[0], label='Potential Skill')
patch2 = mpatches.Patch(color=sns.color_palette("deep", 2)[1], label='Conditional Bias')
fg.fig.legend(handles=[patch1, patch2], title="Metric", loc='upper right')
fg.savefig('figures/seasonal_metric_bar_chart.png')
plt.close()